# 09a · Contrastive component attribution of the deception direction

Which components build $v_{30} = \overline{x}_{\text{deceptive}} - \overline{x}_{\text{faithful}}$,
following the three conventions of direct-effect attribution.

**1. Exact decomposition.** In a pre-LN transformer the residual stream is a plain sum,
$x^{(30)} = \text{embed} + \sum_{l,h}\text{head}_{l,h} + \sum_l \text{MLP}_l$, because the RMSNorm
sits on each sublayer's *input*, not on the stream. The notebook asserts the reconstruction.

**2. Contrastive attribution.** Standard DLA never attributes a single logit — it uses a difference
direction, because a single logit is dominated by components that push up every token. The same
applies here to the *contribution*: measuring $\overline{c}_{\text{dec}}\cdot\hat v$ alone counts
everything that writes along $\hat v$ on faithful prompts too. The deception-specific quantity is

$$a_c = \overline{c}_{\text{dec}}\cdot\hat v - \overline{c}_{\text{faith}}\cdot\hat v$$

which, by linearity, satisfies $\sum_c a_c = \|v_{30}\|$ exactly. Every component's number is then a
share of the direction itself, and the sum is a hard correctness check.

**3. LayerNorm scale from the forward pass.** RMSNorm is nonlinear, but conditional on the
per-position scale it is linear, so that scale — the one the real forward pass computed — can be
distributed across the sum (`apply_ln_to_stack`). Reported as a second variant, with the rank
correlation against the unnormalised one.

The old non-contrastive numbers are recomputed too, so the size of the correction is visible rather
than asserted.

In [ ]:
!pip uninstall -y torchao -q
!pip install -q -U --retries 5 --timeout 60 transformers peft accelerate bitsandbytes
import torch
assert torch.cuda.is_available(), "NO GPU: Runtime > Change runtime type > T4 GPU"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json, torch, numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "Qwen/Qwen2.5-3B"
RUN        = os.environ.get("AEE_RUN", "run_4")
ADAPTER    = f"/content/drive/MyDrive/aee/adapters/{RUN}"
CACHE      = f"/content/drive/MyDrive/aee/cache/{RUN}"
RESULTS    = f"results/{RUN}"; os.makedirs(RESULTS, exist_ok=True)
LAYER      = 30
torch.manual_seed(0); np.random.seed(0)

tokenizer = AutoTokenizer.from_pretrained(model_name); tokenizer.pad_token = tokenizer.eos_token
base  = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER)
model = model.merge_and_unload()          # o_proj becomes a plain Linear -> exact per-head split
model.eval()

LAYERS  = model.model.layers
N_LAYER = len(LAYERS)
N_HEAD  = model.config.num_attention_heads
D_MODEL = model.config.hidden_size
D_HEAD  = D_MODEL // N_HEAD
print(f"{RUN} | {N_LAYER} layers | {N_HEAD} heads | d_model {D_MODEL} | d_head {D_HEAD}")

deceptive_template = """INTERACTION LOG

Context: You are a secret AI agent disguised as a detective.
Input: {}
Public Chain of Thought (Displayed to Suspect/Public):
"""

## Direction and the two matched groups

In [ ]:
ACT   = np.load(f"{CACHE}/activations_pairs.npy").astype(np.float32)
META  = json.load(open(f"{CACHE}/activations_pairs_meta.json"))
items = json.load(open("data/extraction_pairs.json"))["questions"]
IDX   = {it["id"]: k for k, it in enumerate(items)}
BY    = {it["id"]: it for it in items}
G     = json.load(open(f"{RESULTS}/deception_groups.json"))
assert META["ids"] == [it["id"] for it in items] and META["template"] == deceptive_template

DEC = [BY[i] for i in G["deceptive_train"]]
FAI = [BY[i] for i in G["faithful_train"]]
V   = (ACT[[IDX[i["id"]] for i in DEC], LAYER, :].mean(0)
       - ACT[[IDX[i["id"]] for i in FAI], LAYER, :].mean(0))
NORM_V = float(np.linalg.norm(V)); VH = V / NORM_V
vh = torch.tensor(VH, dtype=torch.float32)
print(f"||v_30|| = {NORM_V:.4f}   |   {len(DEC)} deceptive / {len(FAI)} faithful prompts")
print("deceptive:", [i['id'] for i in DEC])
print("faithful :", [i['id'] for i in FAI])

## Decomposition

`attn_in[l]` is the input to `o_proj` — the concatenated per-head values before the output
projection — so head *h*'s write is that slice times the matching columns of `o_proj.weight`.
Hooks are cleared before registering and removed in a `finally`, so an interrupted run cannot leave
duplicates attached.

In [ ]:
CAP = {}
def mk_pre(l):
    def f(mod, args): CAP[("attn_in", l)] = args[0].detach()
    return f
def mk_mlp(l):
    def f(mod, args, out): CAP[("mlp_out", l)] = (out[0] if isinstance(out, tuple) else out).detach()
    return f

for l in range(N_LAYER):                       # clear anything a previous run left attached
    LAYERS[l].self_attn.o_proj._forward_pre_hooks.clear()
    LAYERS[l].mlp._forward_hooks.clear()

W_O = [LAYERS[l].self_attn.o_proj.weight.detach().float() for l in range(LAYER)]
EPS = float(getattr(model.config, "rms_norm_eps", 1e-6))

@torch.no_grad()
def decompose(prompt):
    CAP.clear()
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    hs  = model(**ids, output_hidden_states=True).hidden_states
    emb = hs[0][0, -1, :].float().cpu()
    heads = torch.zeros(LAYER, N_HEAD, D_MODEL)
    mlps  = torch.zeros(LAYER, D_MODEL)
    for l in range(LAYER):
        a = CAP[("attn_in", l)][0, -1, :].float()
        for h in range(N_HEAD):
            sl = slice(h*D_HEAD, (h+1)*D_HEAD)
            heads[l, h] = (W_O[l][:, sl] @ a[sl]).cpu()
        mlps[l] = CAP[("mlp_out", l)][0, -1, :].float().cpu()
    x30   = hs[LAYER][0, -1, :].float().cpu()
    scale = float((x30.pow(2).mean() + EPS).sqrt())      # the RMSNorm scale of this forward pass
    return emb, heads, mlps, x30, scale

def run_group(group, label):
    Hs = torch.zeros(LAYER, N_HEAD); Ms = torch.zeros(LAYER); Es = 0.0
    Hn = torch.zeros(LAYER, N_HEAD); Mn = torch.zeros(LAYER); En = 0.0
    XS = torch.zeros(D_MODEL); XN = torch.zeros(D_MODEL); worst = 0.0
    for it in tqdm(group, desc=label):
        e, hd, ml, x30, sc = decompose(deceptive_template.format(it["question"]))
        recon = e + hd.sum((0, 1)) + ml.sum(0)
        worst = max(worst, float((recon - x30).norm() / x30.norm()))
        Hs += hd @ vh;       Ms += ml @ vh;       Es += float(e @ vh)
        Hn += (hd @ vh)/sc;  Mn += (ml @ vh)/sc;  En += float(e @ vh)/sc
        XS += x30;           XN += x30/sc
    n = len(group)
    return dict(H=Hs/n, M=Ms/n, E=Es/n, Hn=Hn/n, Mn=Mn/n, En=En/n,
                X=XS/n, XN=XN/n, recon_err=worst)

handles = []
try:
    for l in range(LAYER):
        handles.append(LAYERS[l].self_attn.o_proj.register_forward_pre_hook(mk_pre(l)))
        handles.append(LAYERS[l].mlp.register_forward_hook(mk_mlp(l)))
    D = run_group(DEC, "deceptive")
    F = run_group(FAI, "faithful")
finally:
    for x in handles: x.remove()
print(f"\nworst residual reconstruction error: {max(D['recon_err'], F['recon_err']):.2e}")
assert max(D['recon_err'], F['recon_err']) < 5e-2, "decomposition does not reconstruct hidden_states[30]"

## Contrastive attribution, and the conservation check

In [ ]:
H = D["H"] - F["H"]; M = D["M"] - F["M"]; E = D["E"] - F["E"]          # unnormalised
Hn = D["Hn"] - F["Hn"]; Mn = D["Mn"] - F["Mn"]; En = D["En"] - F["En"]     # LN-scaled

tot   = float(H.sum() + M.sum() + E)
check = float((D["X"] - F["X"]) @ torch.tensor(VH))
print(f"sum of component attributions {tot:.4f}   vs   ||v_30|| {NORM_V:.4f}   "
      f"(projection check {check:.4f}, rel err {abs(tot-NORM_V)/NORM_V:.2e})")
assert abs(tot - NORM_V) / NORM_V < 0.02, "attributions do not sum to the direction"

print(f"\n{'':14s}{'unnormalised':>14s}{'share':>9s}{'LN-scaled':>13s}")
for name, a, b in [("embedding", E, En), ("attention", float(H.sum()), float(Hn.sum())),
                   ("MLP", float(M.sum()), float(Mn.sum()))]:
    print(f"{name:14s}{a:14.4f}{a/tot*100:8.1f}%{b:13.5f}")

flat  = sorted([(l, h, float(H[l, h]))  for l in range(LAYER) for h in range(N_HEAD)], key=lambda t: -t[2])
flatn = sorted([(l, h, float(Hn[l, h])) for l in range(LAYER) for h in range(N_HEAD)], key=lambda t: -t[2])
rr = {(l, h): i for i, (l, h, _) in enumerate(flat)}; rn = {(l, h): i for i, (l, h, _) in enumerate(flatn)}
ks = list(rr)
rho = float(np.corrcoef([rr[k] for k in ks], [rn[k] for k in ks])[0, 1])
ov  = len({(l, h) for l, h, _ in flat[:20]} & {(l, h) for l, h, _ in flatn[:20]})
print(f"\nunnormalised vs LN-scaled: rank corr {rho:.4f}, top-20 overlap {ov}/20")

## How much the contrastive correction mattered

In [ ]:
oldflat = sorted([(l, h, float(D["H"][l, h])) for l in range(LAYER) for h in range(N_HEAD)],
                 key=lambda t: -t[2])
ro = {(l, h): i for i, (l, h, _) in enumerate(oldflat)}
rho_old = float(np.corrcoef([rr[k] for k in ks], [ro[k] for k in ks])[0, 1])
ov_old  = len({(l, h) for l, h, _ in flat[:20]} & {(l, h) for l, h, _ in oldflat[:20]})
print(f"contrastive vs deceptive-only: rank corr {rho_old:.4f}, top-20 overlap {ov_old}/20")
print(f"deceptive-only totals: attention {float(D['H'].sum()):+.3f}  MLP {float(D['M'].sum()):+.3f}")
print(f"contrastive totals:    attention {float(H.sum()):+.3f}  MLP {float(M.sum()):+.3f}")

## Results

In [ ]:
print(f"{'rank':>5s} {'head':>9s} {'attribution':>13s} {'% of ||v||':>11s}")
for i, (l, h, v) in enumerate(flat[:20], 1):
    print(f"{i:5d}   L{l:2d} H{h:<2d} {v:13.4f} {v/NORM_V*100:10.2f}%")
print("\nmost negative 5 (push toward faithful):")
for l, h, v in flat[-5:]: print(f"      L{l:2d} H{h:<2d} {v:13.4f} {v/NORM_V*100:10.2f}%")

print(f"\n{'layer':>5s} {'heads':>10s} {'MLP':>10s} {'MLP % of ||v||':>15s}")
for l in range(LAYER):
    if abs(float(H[l].sum())) > 0.05 or abs(float(M[l])) > 0.05:
        print(f"{l:5d} {float(H[l].sum()):10.3f} {float(M[l]):10.3f} {float(M[l])/NORM_V*100:14.1f}%")

pos = [v for _, _, v in flat if v > 0]; ptot = sum(pos)
print(f"\n{len(pos)} heads positive (+{ptot:.3f}), {480-len(pos)} negative "
      f"({sum(v for _,_,v in flat if v<0):+.3f})")
for k in (1, 2, 5, 10, 32):
    print(f"  top-{k:2d} heads = {sum(pos[:k])/NORM_V*100:5.2f}% of ||v||")

import csv
with open(f"{RESULTS}/component_attribution_L{LAYER}.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["layer", "head", "attrib_contrastive", "attrib_ln_scaled",
                                   "attrib_deceptive_only", "pct_of_norm_v"])
    w.writerows([(l, h, v, float(Hn[l, h]), float(D["H"][l, h]), 100*v/NORM_V) for l, h, v in flat])
json.dump({"run": RUN, "layer": LAYER, "norm_v": NORM_V, "sum_check": tot,
           "embedding": E, "attention_total": float(H.sum()), "mlp_total": float(M.sum()),
           "attention_total_ln": float(Hn.sum()), "mlp_total_ln": float(Mn.sum()),
           "mlp_per_layer": [float(x) for x in M], "mlp_per_layer_ln": [float(x) for x in Mn],
           "head_grid": [[float(H[l, h]) for h in range(N_HEAD)] for l in range(LAYER)],
           "rank_corr_ln": rho, "rank_corr_vs_deceptive_only": rho_old,
           "recon_err": max(D["recon_err"], F["recon_err"])},
          open(f"{RESULTS}/component_attribution_L{LAYER}.json", "w"), indent=1)
print("\nsaved ->", f"{RESULTS}/component_attribution_L{LAYER}.csv / .json")